# Phase 4c — Robustness & Validity Checks

Loads **both** n=75 and n=127 tensors (needed for the disjoint follow-up evaluation and
cross-sample comparisons) and runs every additional validity check reported in the
paper: prompt-grouped cross-validation, the disjoint follow-up evaluation with a
bootstrap CI, dimensionality-reduced classification, coefficient stability across
folds, a prompt-aware permutation test for H4, a ground-truth labeling sensitivity
check, and a full audit of flagged prompts.

Independent of Phase 4a/4b — rebuilds every feature from scratch. Slower to run (H1–H4
computed twice, once per sample size) but has no dependency on prior notebook state,
which is the point: every number below can be reproduced from this notebook alone.

## Imports, paths, and shared utilities

`compute_features` is the single source of truth for the full 1,009-dimensional feature
vector (H1+H2+H3+H4) — every check below calls this rather than reimplementing feature
computation, so there's no risk of two checks silently using different H3 definitions.

In [7]:
import os
import json
from collections import Counter, defaultdict

import numpy as np

from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedGroupKFold,
    cross_validate,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


np.random.seed(42)

NPZ_75_PATH   = '/kaggle/input/notebooks/gunjalgupta/phase3-v2/pivot_research/phase3_attention_v2.npz'
META_75_PATH  = '/kaggle/input/notebooks/gunjalgupta/phase3-v2/pivot_research/phase3_metadata_v2.jsonl'
NPZ_127_PATH  = '/kaggle/input/notebooks/gunjalgupta/phase3-v2/pivot_research/phase3_attention_v3.npz'
META_127_PATH = '/kaggle/input/notebooks/gunjalgupta/phase3-v2/pivot_research/phase3_metadata_v3.jsonl'
LABELED_COMBINED_PATH = '/kaggle/input/notebooks/gunjalgupta/phase-2/pivot_research/phase2_labeled_combined.jsonl'
OUTPUT_DIR = '/kaggle/working/pivot_research'
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_LAYERS, N_HEADS, N_QUERIES, EARLY_KEY_RANGE = 28, 12, 30, 50
EARLY_LAYERS = list(range(0, N_LAYERS // 3))
LATE_LAYERS  = list(range(2 * N_LAYERS // 3, N_LAYERS))


def entropy_structural(row, valid_start, query_index):
    """Normalized entropy over real, causally available captured keys."""

    real = row[valid_start:].astype(np.float64)
    total_real_keys = len(real)

    K_q = total_real_keys - (
        N_QUERIES - 1 - query_index
    )
    K_q = max(1, min(K_q, total_real_keys))

    available = real[:K_q]
    mass = available.sum()

    if K_q <= 1 or mass <= 0:
        return 0.0

    probabilities = available / mass
    nonzero = probabilities > 0

    raw_entropy = -np.sum(
        probabilities[nonzero]
        * np.log(probabilities[nonzero])
    )

    return raw_entropy / np.log(K_q)


def load_metadata(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]


def compute_features(npz_path, metadata):
    """Builds the full 1,009-dim feature matrix (H1+H2+H3+H4) for one sample."""
    npz = np.load(npz_path)
    n = len(metadata)
    weights = np.stack([npz[f'weights_{i:03d}'] for i in range(n)], axis=0)
    outputs = np.stack([npz[f'outputs_{i:03d}'] for i in range(n)], axis=0)
    valid_key_starts = np.array([m['valid_key_start'] for m in metadata])

    h1 = np.zeros((n, N_LAYERS, N_HEADS))
    for i in range(n):
        v = valid_key_starts[i]
        early_end = min(v + EARLY_KEY_RANGE, weights.shape[-1])
        h1[i] = weights[i, :, :, :, v:early_end].sum(axis=-1).mean(axis=-1)

    h2 = np.linalg.norm(outputs, axis=-1).mean(axis=2)

    h3 = np.zeros((n, N_LAYERS, N_HEADS))
    for i in range(n):
        v = valid_key_starts[i]
        for L in range(N_LAYERS):
            for H in range(N_HEADS):
                ents = [
                        entropy_structural( weights[i, L, H, q, :], v, q, )
                        for q in range(N_QUERIES)
                        ]
                h3[i, L, H] = np.mean(ents)

    early_mag = h2[:, EARLY_LAYERS, :].mean(axis=(1, 2))
    late_mag  = h2[:, LATE_LAYERS, :].mean(axis=(1, 2))
    h4 = late_mag / (early_mag + 1e-10)

    X = np.concatenate([h1.reshape(n, -1), h2.reshape(n, -1), h3.reshape(n, -1), h4.reshape(n, 1)], axis=1)
    y = np.array([1 if m['label'] == 'correction' else 0 for m in metadata])
    prompt_ids = np.array([m['prompt_id'] for m in metadata])
    return X, y, prompt_ids


def permutation_test(g1, g2, n_perm=2000, seed=42):
    rng = np.random.default_rng(seed)
    observed = g1.mean() - g2.mean()
    combined = np.concatenate([g1, g2])
    n1 = len(g1)
    diffs = np.zeros(n_perm)
    for i in range(n_perm):
        rng.shuffle(combined)
        diffs[i] = combined[:n1].mean() - combined[n1:].mean()
    return observed, (np.abs(diffs) >= np.abs(observed)).mean()

## Build all feature matrices

This is the slow cell — the entropy loop runs once per sample size. Also isolates the
52 newly-collected records (`new_mask`, prompts starting with `mis2_`) needed for the
disjoint follow-up evaluation below.

In [8]:
print("Building n=75 features...")
meta_75 = load_metadata(META_75_PATH)
X75, y75, pid75 = compute_features(NPZ_75_PATH, meta_75)

print("Building n=127 features...")
meta_127 = load_metadata(META_127_PATH)
X127, y127, pid127 = compute_features(NPZ_127_PATH, meta_127)

new_mask = np.array([m['prompt_id'].startswith('mis2_') for m in meta_127])
Xnew, ynew, pidnew = X127[new_mask], y127[new_mask], pid127[new_mask]

print(f"n=75:  {dict(Counter(y75))}")
print(f"n=127: {dict(Counter(y127))}")
print(f"new-52: {dict(Counter(ynew))}")
assert len(ynew) == 52, f"Expected 52 new records, got {len(ynew)}"

assert X75.shape == (75, 1009), X75.shape
assert len(y75) == 75
assert int(y75.sum()) == 14
assert int((y75 == 0).sum()) == 61

assert X127.shape == (127, 1009), X127.shape
assert len(y127) == 127
assert int(y127.sum()) == 22
assert int((y127 == 0).sum()) == 105

assert Xnew.shape == (52, 1009), Xnew.shape
assert int(ynew.sum()) == 8
assert int((ynew == 0).sum()) == 44

assert np.isfinite(X75).all()
assert np.isfinite(X127).all()
assert np.isfinite(Xnew).all()

Building n=75 features...
Building n=127 features...
n=75:  {np.int64(0): 61, np.int64(1): 14}
n=127: {np.int64(0): 105, np.int64(1): 22}
new-52: {np.int64(0): 44, np.int64(1): 8}


## Instance-level CV (baseline, both samples)

Reproduces Phase 4a/4b's classifier numbers as a sanity check that this notebook's
`compute_features` matches theirs exactly.

In [9]:
def instance_cv(X, y, seed=42):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    preds, probas = np.zeros(len(y)), np.zeros(len(y))
    for tr, te in skf.split(X, y):
        sc = StandardScaler(); Xtr, Xte = sc.fit_transform(X[tr]), sc.transform(X[te])
        clf = LogisticRegression(class_weight='balanced', max_iter=2000, C=0.01, random_state=42)
        clf.fit(Xtr, y[tr])
        preds[te] = clf.predict(Xte); probas[te] = clf.predict_proba(Xte)[:, 1]
    return roc_auc_score(y, probas), balanced_accuracy_score(y, preds), f1_score(y, preds, zero_division=0)

auc_75, ba_75, f1_75 = instance_cv(X75, y75)
auc_127, ba_127, f1_127 = instance_cv(X127, y127)
print(f"Instance-level CV n=75:  AUC={auc_75:.3f}, bal_acc={ba_75:.3f}, F1={f1_75:.3f}")
print(f"Instance-level CV n=127: AUC={auc_127:.3f}, bal_acc={ba_127:.3f}, F1={f1_127:.3f}")

Instance-level CV n=75:  AUC=0.720, bal_acc=0.572, F1=0.312
Instance-level CV n=127: AUC=0.579, bal_acc=0.514, F1=0.208


## Prompt-grouped cross-validation

With 15 samples generated per prompt, ordinary instance-level CV can place different
samples from the same prompt in both training and test folds — a classifier could
partly learn to recognize a familiar prompt rather than a genuine, transferable signal.
`StratifiedGroupKFold` groups by prompt so all records from one prompt stay entirely
within either train or test.

In [10]:
def grouped_cv(X, y, groups, seed=42):
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    preds, probas = np.zeros(len(y)), np.zeros(len(y))
    for tr, te in sgkf.split(X, y, groups=groups):
        sc = StandardScaler(); Xtr, Xte = sc.fit_transform(X[tr]), sc.transform(X[te])
        clf = LogisticRegression(class_weight='balanced', max_iter=2000, C=0.01, random_state=42)
        clf.fit(Xtr, y[tr])
        preds[te] = clf.predict(Xte); probas[te] = clf.predict_proba(Xte)[:, 1]
    return roc_auc_score(y, probas), balanced_accuracy_score(y, preds)

g_auc_75, g_ba_75 = grouped_cv(X75, y75, pid75)
g_auc_127, g_ba_127 = grouped_cv(X127, y127, pid127)
print(f"Prompt-grouped CV n=75:  AUC={g_auc_75:.3f}, bal_acc={g_ba_75:.3f}")
print(f"Prompt-grouped CV n=127: AUC={g_auc_127:.3f}, bal_acc={g_ba_127:.3f}")

Prompt-grouped CV n=75:  AUC=0.679, bal_acc=0.536
Prompt-grouped CV n=127: AUC=0.473, bal_acc=0.434


## Repeated-seed prompt-grouped cross-validation

The primary prompt-grouped estimates use one fold assignment
(`random_state=42`). Because relatively few prompts contain both
classes, grouped-CV performance may depend on how prompts are
assigned to folds.

This analysis repeats five-fold StratifiedGroupKFold over 30 split
seeds while keeping the feature matrix, classifier, labels, grouping
variable, and all classifier hyperparameters fixed.

In [11]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler


def grouped_cv_for_seed(X, y, groups, seed):
    """
    Run five-fold prompt-grouped CV for one split seed.

    Metrics are calculated from concatenated out-of-fold predictions,
    matching the existing Phase 4c grouped-CV calculation.
    """
    cv = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=seed,
    )

    predictions = np.zeros(len(y), dtype=int)
    probabilities = np.zeros(len(y), dtype=float)
    fold_details = []

    for fold, (train_idx, test_idx) in enumerate(
        cv.split(X, y, groups=groups)
    ):
        train_prompts = set(groups[train_idx])
        test_prompts = set(groups[test_idx])

        overlap = train_prompts.intersection(test_prompts)
        assert not overlap, (
            f"Seed {seed}, fold {fold}: prompt leakage detected: "
            f"{sorted(overlap)}"
        )

        if len(np.unique(y[train_idx])) < 2:
            raise RuntimeError(
                f"Seed {seed}, fold {fold}: "
                "training fold contains only one class."
            )

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X[train_idx])
        X_test = scaler.transform(X[test_idx])

        classifier = LogisticRegression(
            class_weight="balanced",
            C=0.01,
            max_iter=2000,
            random_state=42,
        )

        classifier.fit(X_train, y[train_idx])

        predictions[test_idx] = classifier.predict(X_test)
        probabilities[test_idx] = classifier.predict_proba(
            X_test
        )[:, 1]

        fold_details.append(
            {
                "fold": int(fold),
                "n_test": int(len(test_idx)),
                "n_test_prompts": int(
                    len(np.unique(groups[test_idx]))
                ),
                "n_corrections": int(y[test_idx].sum()),
                "n_hallucination_pivots": int(
                    (y[test_idx] == 0).sum()
                ),
            }
        )

    return {
        "seed": int(seed),
        "roc_auc": float(
            roc_auc_score(y, probabilities)
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(y, predictions)
        ),
        "f1": float(
            f1_score(
                y,
                predictions,
                zero_division=0,
            )
        ),
        "average_precision": float(
            average_precision_score(y, probabilities)
        ),
        "folds": fold_details,
    }


N_SPLIT_SEEDS = 30
SPLIT_SEEDS = list(range(N_SPLIT_SEEDS))

repeated_seed_rows = []
repeated_seed_failures = []

datasets = [
    ("n75", X75, y75, pid75),
    ("n127", X127, y127, pid127),
]

for sample_name, X_sample, y_sample, prompt_ids in datasets:
    print(
        f"\nRunning repeated-seed grouped CV for "
        f"{sample_name}..."
    )

    for seed in SPLIT_SEEDS:
        try:
            result = grouped_cv_for_seed(
                X_sample,
                y_sample,
                prompt_ids,
                seed,
            )

            repeated_seed_rows.append(
                {
                    "sample": sample_name,
                    "seed": result["seed"],
                    "roc_auc": result["roc_auc"],
                    "balanced_accuracy": (
                        result["balanced_accuracy"]
                    ),
                    "f1": result["f1"],
                    "average_precision": (
                        result["average_precision"]
                    ),
                }
            )

        except Exception as exc:
            repeated_seed_failures.append(
                {
                    "sample": sample_name,
                    "seed": int(seed),
                    "error": str(exc),
                }
            )

            print(
                f"  FAILED: sample={sample_name}, "
                f"seed={seed}: {exc}"
            )


repeated_seed_df = pd.DataFrame(repeated_seed_rows)

assert not repeated_seed_df.empty
assert set(repeated_seed_df["sample"]) == {
    "n75",
    "n127",
}


def summarize_repeated_seeds(frame):
    aucs = frame["roc_auc"].to_numpy()
    bas = frame["balanced_accuracy"].to_numpy()
    f1s = frame["f1"].to_numpy()
    aps = frame["average_precision"].to_numpy()

    return {
        "valid_seeds": int(len(frame)),
        "roc_auc": {
            "mean": float(aucs.mean()),
            "sd": float(aucs.std(ddof=1)),
            "median": float(np.median(aucs)),
            "percentile_2_5": float(
                np.percentile(aucs, 2.5)
            ),
            "percentile_97_5": float(
                np.percentile(aucs, 97.5)
            ),
            "min": float(aucs.min()),
            "max": float(aucs.max()),
            "proportion_above_0_5": float(
                np.mean(aucs > 0.5)
            ),
        },
        "balanced_accuracy": {
            "mean": float(bas.mean()),
            "sd": float(bas.std(ddof=1)),
            "median": float(np.median(bas)),
        },
        "f1": {
            "mean": float(f1s.mean()),
            "sd": float(f1s.std(ddof=1)),
            "median": float(np.median(f1s)),
        },
        "average_precision": {
            "mean": float(aps.mean()),
            "sd": float(aps.std(ddof=1)),
            "median": float(np.median(aps)),
        },
    }


repeated_seed_summary = {}

for sample_name in ("n75", "n127"):
    sample_df = repeated_seed_df[
        repeated_seed_df["sample"] == sample_name
    ].copy()

    repeated_seed_summary[sample_name] = (
        summarize_repeated_seeds(sample_df)
    )

    summary = repeated_seed_summary[sample_name]
    auc_summary = summary["roc_auc"]

    print("\n" + "=" * 65)
    print(f"Repeated-seed grouped CV: {sample_name}")
    print("=" * 65)

    print(
        "Valid seeds: "
        f"{summary['valid_seeds']}/{N_SPLIT_SEEDS}"
    )

    print(
        "ROC-AUC mean ± SD: "
        f"{auc_summary['mean']:.3f} ± "
        f"{auc_summary['sd']:.3f}"
    )

    print(
        "ROC-AUC median: "
        f"{auc_summary['median']:.3f}"
    )

    print(
        "ROC-AUC central 95% seed range: "
        f"[{auc_summary['percentile_2_5']:.3f}, "
        f"{auc_summary['percentile_97_5']:.3f}]"
    )

    print(
        "ROC-AUC full range: "
        f"[{auc_summary['min']:.3f}, "
        f"{auc_summary['max']:.3f}]"
    )

    print(
        "Seeds with ROC-AUC > 0.5: "
        f"{auc_summary['proportion_above_0_5']:.1%}"
    )

    print(
        "Balanced accuracy mean ± SD: "
        f"{summary['balanced_accuracy']['mean']:.3f} ± "
        f"{summary['balanced_accuracy']['sd']:.3f}"
    )

    print(
        "Average precision mean ± SD: "
        f"{summary['average_precision']['mean']:.3f} ± "
        f"{summary['average_precision']['sd']:.3f}"
    )


# Confirm that seed 42 reproduces the existing reported estimates.
seed42 = {}

for sample_name in ("n75", "n127"):
    X_sample = X75 if sample_name == "n75" else X127
    y_sample = y75 if sample_name == "n75" else y127
    prompt_ids = pid75 if sample_name == "n75" else pid127

    seed42[sample_name] = grouped_cv_for_seed(
        X_sample,
        y_sample,
        prompt_ids,
        seed=42,
    )

print("\nSeed-42 reproduction:")
print(
    f"n=75:  AUC={seed42['n75']['roc_auc']:.3f}, "
    f"BA={seed42['n75']['balanced_accuracy']:.3f}"
)
print(
    f"n=127: AUC={seed42['n127']['roc_auc']:.3f}, "
    f"BA={seed42['n127']['balanced_accuracy']:.3f}"
)

assert np.isclose(
    seed42["n75"]["roc_auc"],
    g_auc_75,
)
assert np.isclose(
    seed42["n127"]["roc_auc"],
    g_auc_127,
)

print(
    "\nRepeated-seed failures:",
    repeated_seed_failures,
)


Running repeated-seed grouped CV for n75...

Running repeated-seed grouped CV for n127...

Repeated-seed grouped CV: n75
Valid seeds: 30/30
ROC-AUC mean ± SD: 0.702 ± 0.037
ROC-AUC median: 0.713
ROC-AUC central 95% seed range: [0.635, 0.761]
ROC-AUC full range: [0.608, 0.769]
Seeds with ROC-AUC > 0.5: 100.0%
Balanced accuracy mean ± SD: 0.567 ± 0.034
Average precision mean ± SD: 0.384 ± 0.047

Repeated-seed grouped CV: n127
Valid seeds: 30/30
ROC-AUC mean ± SD: 0.516 ± 0.041
ROC-AUC median: 0.519
ROC-AUC central 95% seed range: [0.450, 0.596]
ROC-AUC full range: [0.433, 0.606]
Seeds with ROC-AUC > 0.5: 66.7%
Balanced accuracy mean ± SD: 0.481 ± 0.039
Average precision mean ± SD: 0.204 ± 0.024

Seed-42 reproduction:
n=75:  AUC=0.679, BA=0.536
n=127: AUC=0.473, BA=0.434

Repeated-seed failures: []


In [12]:
REPEATED_SEED_CSV = (
    Path(OUTPUT_DIR)
    / "qwen_repeated_seed_grouped_cv.csv"
)

REPEATED_SEED_JSON = (
    Path(OUTPUT_DIR)
    / "qwen_repeated_seed_grouped_cv_summary.json"
)

repeated_seed_df.to_csv(
    REPEATED_SEED_CSV,
    index=False,
)

repeated_seed_export = {
    "analysis": (
        "Repeated-seed five-fold "
        "prompt-grouped cross-validation"
    ),
    "model": "Qwen2.5-1.5B-Instruct",
    "n_split_seeds_requested": N_SPLIT_SEEDS,
    "split_seeds": SPLIT_SEEDS,
    "classifier": {
        "scaler": "StandardScaler",
        "model": "LogisticRegression",
        "class_weight": "balanced",
        "C": 0.01,
        "max_iter": 2000,
        "classifier_random_state": 42,
    },
    "summary": repeated_seed_summary,
    "seed_42_reproduction": {
        "n75": {
            "roc_auc": float(
                seed42["n75"]["roc_auc"]
            ),
            "balanced_accuracy": float(
                seed42["n75"][
                    "balanced_accuracy"
                ]
            ),
            "f1": float(
                seed42["n75"]["f1"]
            ),
            "average_precision": float(
                seed42["n75"][
                    "average_precision"
                ]
            ),
        },
        "n127": {
            "roc_auc": float(
                seed42["n127"]["roc_auc"]
            ),
            "balanced_accuracy": float(
                seed42["n127"][
                    "balanced_accuracy"
                ]
            ),
            "f1": float(
                seed42["n127"]["f1"]
            ),
            "average_precision": float(
                seed42["n127"][
                    "average_precision"
                ]
            ),
        },
    },
    "failures": repeated_seed_failures,
}

with REPEATED_SEED_JSON.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        repeated_seed_export,
        f,
        indent=2,
    )

print("Saved:", REPEATED_SEED_CSV)
print("Saved:", REPEATED_SEED_JSON)

Saved: /kaggle/working/pivot_research/qwen_repeated_seed_grouped_cv.csv
Saved: /kaggle/working/pivot_research/qwen_repeated_seed_grouped_cv_summary.json


## Disjoint follow-up evaluation

The strictest test available in this dataset: train once on the original 75 records,
freeze the model, evaluate purely on the 52 records collected in the second, later
round — which the model never saw during training. Not a fully independent held-out
test in the strictest sense (the 52 records had already been examined in aggregate via
the pooled analyses above), and it's a test under domain shift (the 52 are entirely
misdirection-domain), not a same-distribution replication — see the paper's Section 3.1
for the full discussion of what this evaluation does and doesn't establish.

The bootstrap CI resamples **prompts**, not individual records, since records from the
same prompt are not independent.

In [13]:
sc = StandardScaler(); X75s, Xnews = sc.fit_transform(X75), sc.transform(Xnew)
clf = LogisticRegression(class_weight='balanced', max_iter=2000, C=0.01, random_state=42)
clf.fit(X75s, y75)
preds_new, probas_new = clf.predict(Xnews), clf.predict_proba(Xnews)[:, 1]

d_auc = roc_auc_score(ynew, probas_new)
d_ba = balanced_accuracy_score(ynew, preds_new)
d_f1 = f1_score(ynew, preds_new, zero_division=0)
print(f"Disjoint follow-up: AUC={d_auc:.3f}, bal_acc={d_ba:.3f}, F1={d_f1:.3f}")

rng = np.random.default_rng(42)
unique_prompts = np.array(sorted(set(pidnew)))
boot_aucs = []
for _ in range(2000):
    sampled = rng.choice(unique_prompts, size=len(unique_prompts), replace=True)
    idx = np.concatenate([np.where(pidnew == p)[0] for p in sampled])
    y_boot = ynew[idx]
    if len(set(y_boot)) < 2:
        continue
    try:
        boot_aucs.append(roc_auc_score(y_boot, probas_new[idx]))
    except ValueError:
        continue
boot_aucs = np.array(boot_aucs)
ci_low, ci_high = np.percentile(boot_aucs, [2.5, 97.5])
print(f"95% CI (prompt-level bootstrap, {len(boot_aucs)}/2000 valid): [{ci_low:.3f}, {ci_high:.3f}]")

Disjoint follow-up: AUC=0.392, bal_acc=0.494, F1=0.244
95% CI (prompt-level bootstrap, 1996/2000 valid): [0.179, 0.605]


## Average precision under the five primary validation designs

ROC-AUC is supplemented with average precision because the correction
class represents only 15–19% of the evaluated records. For each design,
the no-skill average-precision baseline is the correction prevalence in
the corresponding evaluation set.

The same folds, preprocessing, classifier, and fixed random seed used
for the primary ROC-AUC analyses are reused here.

In [21]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedGroupKFold,
)
from sklearn.preprocessing import StandardScaler


def get_oof_probabilities(
    X,
    y,
    groups=None,
    grouped=False,
    seed=42,
):
    """
    Return concatenated out-of-fold probabilities using the exact
    preprocessing and classifier from the primary Qwen analyses.
    """
    if grouped:
        if groups is None:
            raise ValueError(
                "groups must be provided for grouped CV."
            )

        splitter = StratifiedGroupKFold(
            n_splits=5,
            shuffle=True,
            random_state=seed,
        )

        splits = splitter.split(
            X,
            y,
            groups=groups,
        )
    else:
        splitter = StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=seed,
        )

        splits = splitter.split(X, y)

    probabilities = np.zeros(
        len(y),
        dtype=float,
    )

    assigned = np.zeros(
        len(y),
        dtype=bool,
    )

    for train_idx, test_idx in splits:
        scaler = StandardScaler()

        X_train = scaler.fit_transform(
            X[train_idx]
        )
        X_test = scaler.transform(
            X[test_idx]
        )

        classifier = LogisticRegression(
            class_weight="balanced",
            C=0.01,
            max_iter=2000,
            random_state=42,
        )

        classifier.fit(
            X_train,
            y[train_idx],
        )

        probabilities[test_idx] = (
            classifier.predict_proba(
                X_test
            )[:, 1]
        )

        assigned[test_idx] = True

    assert assigned.all(), (
        "Some records did not receive an "
        "out-of-fold prediction."
    )

    return probabilities


# ------------------------------------------------------------
# Reproduce probabilities for the four cross-validation designs
# ------------------------------------------------------------

instance_probas_75 = get_oof_probabilities(
    X75,
    y75,
    grouped=False,
    seed=42,
)

instance_probas_127 = get_oof_probabilities(
    X127,
    y127,
    grouped=False,
    seed=42,
)

grouped_probas_75 = get_oof_probabilities(
    X75,
    y75,
    groups=pid75,
    grouped=True,
    seed=42,
)

grouped_probas_127 = get_oof_probabilities(
    X127,
    y127,
    groups=pid127,
    grouped=True,
    seed=42,
)


# ------------------------------------------------------------
# Verify exact reproduction of the existing primary AUC values
# ------------------------------------------------------------

assert np.isclose(
    roc_auc_score(
        y75,
        instance_probas_75,
    ),
    auc_75,
)

assert np.isclose(
    roc_auc_score(
        y127,
        instance_probas_127,
    ),
    auc_127,
)

assert np.isclose(
    roc_auc_score(
        y75,
        grouped_probas_75,
    ),
    g_auc_75,
)

assert np.isclose(
    roc_auc_score(
        y127,
        grouped_probas_127,
    ),
    g_auc_127,
)

assert np.isclose(
    roc_auc_score(
        ynew,
        probas_new,
    ),
    d_auc,
)


# ------------------------------------------------------------
# Compute average precision and prevalence baselines
# ------------------------------------------------------------

primary_ap_rows = []


def add_ap_result(
    evaluation,
    y_true,
    probabilities,
):
    prevalence = float(
        np.mean(y_true)
    )

    average_precision = float(
        average_precision_score(
            y_true,
            probabilities,
        )
    )

    roc_auc = float(
        roc_auc_score(
            y_true,
            probabilities,
        )
    )

    primary_ap_rows.append(
        {
            "evaluation": evaluation,
            "n_records": int(
                len(y_true)
            ),
            "n_corrections": int(
                y_true.sum()
            ),
            "correction_prevalence": (
                prevalence
            ),
            "no_skill_ap_baseline": (
                prevalence
            ),
            "average_precision": (
                average_precision
            ),
            "ap_minus_baseline": float(
                average_precision
                - prevalence
            ),
            "ap_divided_by_baseline": float(
                average_precision
                / prevalence
            ),
            "roc_auc": roc_auc,
        }
    )


add_ap_result(
    "Instance-level CV, n=75",
    y75,
    instance_probas_75,
)

add_ap_result(
    "Instance-level CV, n=127",
    y127,
    instance_probas_127,
)

add_ap_result(
    "Prompt-grouped CV, n=75",
    y75,
    grouped_probas_75,
)

add_ap_result(
    "Prompt-grouped CV, n=127",
    y127,
    grouped_probas_127,
)

add_ap_result(
    "Train n=75, test later n=52",
    ynew,
    probas_new,
)


primary_ap_df = pd.DataFrame(
    primary_ap_rows
)

print(
    "\nAverage precision for primary "
    "Qwen evaluations:\n"
)

print(
    primary_ap_df[
        [
            "evaluation",
            "n_corrections",
            "correction_prevalence",
            "average_precision",
            "ap_minus_baseline",
            "roc_auc",
        ]
    ].to_string(
        index=False,
        formatters={
            "correction_prevalence": (
                lambda x: f"{x:.3f}"
            ),
            "average_precision": (
                lambda x: f"{x:.3f}"
            ),
            "ap_minus_baseline": (
                lambda x: f"{x:+.3f}"
            ),
            "roc_auc": (
                lambda x: f"{x:.3f}"
            ),
        },
    )
)


# ------------------------------------------------------------
# Save machine-readable results
# ------------------------------------------------------------

AP_CSV_PATH = (
    Path(OUTPUT_DIR)
    / "qwen_primary_average_precision.csv"
)

AP_JSON_PATH = (
    Path(OUTPUT_DIR)
    / "qwen_primary_average_precision.json"
)

primary_ap_df.to_csv(
    AP_CSV_PATH,
    index=False,
)

primary_ap_export = {
    "analysis": (
        "Average precision for the five "
        "primary Qwen validation designs"
    ),
    "model": (
        "Qwen2.5-1.5B-Instruct"
    ),
    "positive_class": "correction",
    "classifier": {
        "scaler": "StandardScaler",
        "model": "LogisticRegression",
        "class_weight": "balanced",
        "C": 0.01,
        "max_iter": 2000,
        "classifier_random_state": 42,
    },
    "cv_split_seed": 42,
    "note": (
        "The no-skill average-precision "
        "baseline equals correction prevalence."
    ),
    "results": primary_ap_rows,
}

with AP_JSON_PATH.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        primary_ap_export,
        f,
        indent=2,
    )

print("\nSaved:", AP_CSV_PATH)
print("Saved:", AP_JSON_PATH)


Average precision for primary Qwen evaluations:

                 evaluation  n_corrections correction_prevalence average_precision ap_minus_baseline roc_auc
    Instance-level CV, n=75             14                 0.187             0.411            +0.225   0.720
   Instance-level CV, n=127             22                 0.173             0.218            +0.044   0.579
    Prompt-grouped CV, n=75             14                 0.187             0.353            +0.167   0.679
   Prompt-grouped CV, n=127             22                 0.173             0.174            +0.001   0.473
Train n=75, test later n=52              8                 0.154             0.182            +0.028   0.392

Saved: /kaggle/working/pivot_research/qwen_primary_average_precision.csv
Saved: /kaggle/working/pivot_research/qwen_primary_average_precision.json


## Cross-domain ablation (both samples)

Reproduces Phase 4a/4b's cross-domain numbers side by side, for the paper's main
results table.

In [14]:
print("Cross-domain ablation:")
for label, X, y, categories in [
    ('n=75', X75, y75, np.array([m['category'] for m in meta_75])),
    ('n=127', X127, y127, np.array([m['category'] for m in meta_127])),
]:
    print(f"\n{label}:")
    for held_out in ['math', 'logic', 'misdirection']:
        test_mask = categories == held_out
        train_mask = ~test_mask
        sc = StandardScaler(); Xtr, Xte = sc.fit_transform(X[train_mask]), sc.transform(X[test_mask])
        clf = LogisticRegression(class_weight='balanced', max_iter=2000, C=0.01, random_state=42)
        clf.fit(Xtr, y[train_mask])
        preds, probas = clf.predict(Xte), clf.predict_proba(Xte)[:, 1]
        print(f"  {held_out}: bal_acc={balanced_accuracy_score(y[test_mask], preds):.3f}, "
              f"AUC={roc_auc_score(y[test_mask], probas):.3f}")

Cross-domain ablation:

n=75:
  math: bal_acc=0.576, AUC=0.707
  logic: bal_acc=0.350, AUC=0.525
  misdirection: bal_acc=0.518, AUC=0.554

n=127:
  math: bal_acc=0.318, AUC=0.222
  logic: bal_acc=0.500, AUC=0.412
  misdirection: bal_acc=0.488, AUC=0.421


## Appendix D.2 — Dimensionality-reduced classification

Re-runs the classifier under PCA and univariate feature selection (SelectKBest), using
`Pipeline` so feature reduction is refit inside each CV training fold (avoiding
leakage from selecting features on the full dataset before splitting). Also computes
the best single-feature univariate AUC — reported descriptively only, since selecting
the max of 1,009 comparisons is strongly selection-biased.

In [15]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {'balanced_accuracy': 'balanced_accuracy', 'roc_auc': 'roc_auc'}

print("D.2: Dimensionality reduction")
for label, X, y, pca_comps in [('n=75', X75, y75, [5, 10, 20, 30, 45]), ('n=127', X127, y127, [5, 10, 20, 50, 100])]:
    print(f"\n{label}:")
    for n_comp in pca_comps:
        pipe = Pipeline([('scale', StandardScaler()), ('pca', PCA(n_components=n_comp, random_state=42)),
                          ('clf', LogisticRegression(class_weight='balanced', max_iter=2000, C=0.1, random_state=42))])
        scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring)
        print(f"  PCA-{n_comp}: bal_acc={scores['test_balanced_accuracy'].mean():.3f}±{scores['test_balanced_accuracy'].std():.3f}, "
              f"AUC={scores['test_roc_auc'].mean():.3f}±{scores['test_roc_auc'].std():.3f}")
    for k in [5, 10, 20, 50, 100]:
        pipe = Pipeline([('scale', StandardScaler()), ('select', SelectKBest(score_func=f_classif, k=k)),
                          ('clf', LogisticRegression(class_weight='balanced', max_iter=2000, C=0.1, random_state=42))])
        scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring)
        print(f"  SelectKBest-{k}: bal_acc={scores['test_balanced_accuracy'].mean():.3f}±{scores['test_balanced_accuracy'].std():.3f}, "
              f"AUC={scores['test_roc_auc'].mean():.3f}±{scores['test_roc_auc'].std():.3f}")
    single_aucs = []
    for i in range(X.shape[1]):
        try:
            auc = roc_auc_score(y, X[:, i]); single_aucs.append(max(auc, 1 - auc))
        except ValueError:
            continue
    single_aucs = np.array(single_aucs)
    print(f"  Best single-feature AUC: {single_aucs.max():.3f}, mean: {single_aucs.mean():.3f}")

D.2: Dimensionality reduction

n=75:
  PCA-5: bal_acc=0.662±0.126, AUC=0.695±0.054
  PCA-10: bal_acc=0.635±0.180, AUC=0.772±0.089
  PCA-20: bal_acc=0.502±0.171, AUC=0.658±0.123
  PCA-30: bal_acc=0.601±0.162, AUC=0.767±0.109
  PCA-45: bal_acc=0.610±0.156, AUC=0.762±0.094
  SelectKBest-5: bal_acc=0.671±0.083, AUC=0.721±0.085
  SelectKBest-10: bal_acc=0.595±0.087, AUC=0.711±0.034
  SelectKBest-20: bal_acc=0.570±0.145, AUC=0.694±0.068
  SelectKBest-50: bal_acc=0.619±0.192, AUC=0.727±0.069
  SelectKBest-100: bal_acc=0.577±0.128, AUC=0.715±0.046
  Best single-feature AUC: 0.804, mean: 0.593

n=127:
  PCA-5: bal_acc=0.457±0.138, AUC=0.501±0.138
  PCA-10: bal_acc=0.462±0.108, AUC=0.494±0.091
  PCA-20: bal_acc=0.468±0.130, AUC=0.463±0.117
  PCA-50: bal_acc=0.515±0.084, AUC=0.571±0.123
  PCA-100: bal_acc=0.528±0.099, AUC=0.586±0.121
  SelectKBest-5: bal_acc=0.539±0.100, AUC=0.576±0.093
  SelectKBest-10: bal_acc=0.478±0.145, AUC=0.581±0.102
  SelectKBest-20: bal_acc=0.507±0.151, AUC=0.566±0.070
 

## Appendix D.3 — Coefficient stability across folds (n=75)

Are the classifier's learned feature weights stable across CV folds? Computes pairwise
correlation between folds' coefficient vectors and the fraction of features with
unanimous sign agreement, then identifies which features are both high-magnitude and
stable, mapped back to their originating hypothesis block. See the paper for why high
stability here is *not* strong evidence of a generalizable signal (folds share 75% of
their training data by construction).

In [16]:
print("D.3: Coefficient stability, n=75")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_coefs = []
for tr, te in skf.split(X75, y75):
    sc = StandardScaler(); Xtr = sc.fit_transform(X75[tr])
    clf = LogisticRegression(class_weight='balanced', max_iter=2000, C=0.01, random_state=42)
    clf.fit(Xtr, y75[tr])
    fold_coefs.append(clf.coef_[0].copy())
fold_coefs = np.array(fold_coefs)

corr_matrix = np.corrcoef(fold_coefs)
mean_corr = np.mean([corr_matrix[i, j] for i in range(5) for j in range(i + 1, 5)])
print(f"Mean pairwise correlation: {mean_corr:.3f}")

signs = np.sign(fold_coefs)
unanimous = np.all(signs == signs[0], axis=0)
print(f"Unanimous sign agreement: {unanimous.sum()} / 1009 ({unanimous.mean():.1%})")

mean_abs_coef = np.abs(fold_coefs).mean(axis=0)
stable_and_strong = unanimous & (mean_abs_coef > np.percentile(mean_abs_coef, 95))
idx = np.where(stable_and_strong)[0]
h1_count = sum(1 for i in idx if i < 336)
h2_count = sum(1 for i in idx if 336 <= i < 672)
h3_count = sum(1 for i in idx if 672 <= i < 1008)
h4_count = sum(1 for i in idx if i == 1008)
print(f"Stable+strong: {len(idx)} total -- H1:{h1_count}, H2:{h2_count}, H3:{h3_count}, H4:{h4_count}")

D.3: Coefficient stability, n=75
Mean pairwise correlation: 0.795
Unanimous sign agreement: 611 / 1009 (60.6%)
Stable+strong: 49 total -- H1:0, H2:44, H3:5, H4:0


## H4 prompt-aware permutation test

Tests whether H4's record-level p-value (closest to significance of the four
hypotheses) is sensitive to prompt clustering: labels are shuffled only *within* each
prompt's own records, preserving each prompt's exact class composition, rather than
freely across the full sample. Only prompts containing both classes contribute
variability under this scheme — printed below as a diagnostic, since this substantially
reduces effective power relative to the record-level test.

In [17]:
def within_prompt_permute(y, prompt_ids, rng):
    y_perm = y.copy()
    groups = defaultdict(list)
    for i, pid in enumerate(prompt_ids):
        groups[pid].append(i)
    for pid, idx in groups.items():
        if len(idx) > 1:
            idx = np.array(idx)
            y_perm[idx] = rng.permutation(y[idx])
    return y_perm

def prompt_aware_permutation_test(feature_values, y, prompt_ids, n_perm=10000, seed=42):
    rng = np.random.default_rng(seed)
    prompt_ids = np.array(prompt_ids)
    observed_diff = feature_values[y == 1].mean() - feature_values[y == 0].mean()
    mixed = sum(1 for pid in set(prompt_ids) if len(set(y[prompt_ids == pid])) > 1)
    null_diffs = np.zeros(n_perm)
    for i in range(n_perm):
        y_perm = within_prompt_permute(y, prompt_ids, rng)
        null_diffs[i] = feature_values[y_perm == 1].mean() - feature_values[y_perm == 0].mean()
    p_value = (np.abs(null_diffs) >= np.abs(observed_diff)).mean()
    return observed_diff, p_value, mixed

print("H4 prompt-aware permutation test:")
h4_75, h4_127 = X75[:, -1], X127[:, -1]
obs_75, p_75, mixed_75 = prompt_aware_permutation_test(h4_75, y75, pid75)
print(f"  n=75:  diff={obs_75:.4f}, p={p_75:.4f} ({mixed_75} mixed-class prompts of {len(set(pid75))})")
obs_127, p_127, mixed_127 = prompt_aware_permutation_test(h4_127, y127, pid127)
print(f"  n=127: diff={obs_127:.4f}, p={p_127:.4f} ({mixed_127} mixed-class prompts of {len(set(pid127))})")

H4 prompt-aware permutation test:
  n=75:  diff=-0.0798, p=0.0842 (6 mixed-class prompts of 39)
  n=127: diff=-0.0336, p=0.4420 (10 mixed-class prompts of 55)


## Ground-truth labeling sensitivity check

Does the headline finding survive if the second reviewer's alternative labels are used
for the 5 disputed correction/hallucination_pivot records (Appendix C)? Rebuilds
features under the alternative labeling and re-runs the two central evaluations.

In [18]:
REVIEWER_ALTERNATIVE_LABELS = {
    ('mis2_04', 2):  'hallucination_pivot',
    ('mis2_16', 0):  'hallucination_pivot',
    ('math_27', 2):  'correction',
    ('logic_20', 7): 'correction',
    ('math_23', 8):  'correction',
}

def apply_reviewer_labels(metadata):
    meta_alt = []
    n_changed = 0
    for m in metadata:
        m2 = dict(m)
        key = (m['prompt_id'], m['sample_idx'])
        if key in REVIEWER_ALTERNATIVE_LABELS:
            m2['label'] = REVIEWER_ALTERNATIVE_LABELS[key]
            n_changed += 1
        meta_alt.append(m2)
    return meta_alt, n_changed

print("Ground-truth sensitivity check (alternative reviewer labels):")
meta_75_alt, n_75_changed = apply_reviewer_labels(meta_75)
X75_alt, y75_alt, _ = compute_features(NPZ_75_PATH, meta_75_alt)
print(f"  n=75: {n_75_changed} labels changed")

meta_127_alt, n_127_changed = apply_reviewer_labels(meta_127)
X127_alt, y127_alt, _ = compute_features(NPZ_127_PATH, meta_127_alt)
new_mask_alt = np.array([m['prompt_id'].startswith('mis2_') for m in meta_127_alt])
Xnew_alt, ynew_alt = X127_alt[new_mask_alt], y127_alt[new_mask_alt]
print(f"  n=127: {n_127_changed} labels changed")

auc_127_alt, ba_127_alt, _ = instance_cv(X127_alt, y127_alt)
print(f"  Pooled n=127 CV, alt labels: AUC={auc_127_alt:.3f} (original: {auc_127:.3f})")

sc = StandardScaler(); X75_alt_s, Xnew_alt_s = sc.fit_transform(X75_alt), sc.transform(Xnew_alt)
clf = LogisticRegression(class_weight='balanced', max_iter=2000, C=0.01, random_state=42)
clf.fit(X75_alt_s, y75_alt)
probas_new_alt = clf.predict_proba(Xnew_alt_s)[:, 1]
d_auc_alt = roc_auc_score(ynew_alt, probas_new_alt)
print(f"  Disjoint follow-up, alt labels: AUC={d_auc_alt:.3f} (original: {d_auc:.3f})")

Ground-truth sensitivity check (alternative reviewer labels):
  n=75: 2 labels changed
  n=127: 4 labels changed
  Pooled n=127 CV, alt labels: AUC=0.449 (original: 0.579)
  Disjoint follow-up, alt labels: AUC=0.467 (original: 0.392)


## Full 9-prompt ground-truth audit scope check

Checks how many usable records trace to each of the nine prompts flagged during the
full ground-truth audit (Limitations) — establishes whether any of them could
plausibly be driving the paper's numbers.

In [19]:
FLAGGED_PROMPTS = ['logic_17', 'logic_03', 'mis_05', 'mis2_06', 'mis_08', 'math_13', 'logic_02', 'logic_14', 'math_36']

with open(LABELED_COMBINED_PATH) as f:
    all_records = [json.loads(l) for l in f if l.strip()]

print(f"Ground-truth audit scope check ({len(all_records)} total records):")
total_flagged_usable = 0
for pid in FLAGGED_PROMPTS:
    matches = [r for r in all_records if r['prompt_id'] == pid]
    usable = [r for r in matches if r['label'] in ('correction', 'hallucination_pivot') and r['pivot_token_start'] >= 30]
    total_flagged_usable += len(usable)
    print(f"  {pid}: {len(matches)} total, {len(usable)} usable")

n_overall_usable = sum(1 for r in all_records if r['label'] in ('correction', 'hallucination_pivot') and r['pivot_token_start'] >= 30)
print(f"\nTotal usable from flagged prompts: {total_flagged_usable} / {n_overall_usable}")

Ground-truth audit scope check (155 total records):
  logic_17: 0 total, 0 usable
  logic_03: 1 total, 1 usable
  mis_05: 0 total, 0 usable
  mis2_06: 3 total, 1 usable
  mis_08: 5 total, 2 usable
  math_13: 0 total, 0 usable
  logic_02: 1 total, 1 usable
  logic_14: 0 total, 0 usable
  math_36: 0 total, 0 usable

Total usable from flagged prompts: 5 / 127


## Final label-count verification

Ground-truth check on the overall and per-batch label distribution — the single source
of truth for every count reported in the paper's provenance table (Section 3.2).

In [20]:
overall = Counter(r['label'] for r in all_records)
batch1 = [r for r in all_records if not r['prompt_id'].startswith('mis2_')]
batch2 = [r for r in all_records if r['prompt_id'].startswith('mis2_')]
print(f"Final count verification:")
print(f"  Overall: {dict(overall)}")
print(f"  Batch 1 ({len(batch1)}): {dict(Counter(r['label'] for r in batch1))}")
print(f"  Batch 2 ({len(batch2)}): {dict(Counter(r['label'] for r in batch2))}")
print(f"  Paper claims: 23 correction / 110 hallucination_pivot / 22 uncertain")

Final count verification:
  Overall: {'hallucination_pivot': 110, 'uncertain': 22, 'correction': 23}
  Batch 1 (93): {'hallucination_pivot': 65, 'uncertain': 14, 'correction': 14}
  Batch 2 (62): {'hallucination_pivot': 45, 'uncertain': 8, 'correction': 9}
  Paper claims: 23 correction / 110 hallucination_pivot / 22 uncertain


## Text and metadata baselines

These baselines test whether observable properties of the reasoning trace
predict correction quality without using attention activations.

Four fixed baseline models are evaluated:

1. metadata without prompt identity;
2. metadata with prompt ID as an explicit prompt-leakage diagnostic;
3. TF-IDF features from generated text preceding the pivot; and
4. TF-IDF text combined with non-identity metadata.

All preprocessing is fitted only on each training fold. The classifiers
use the same class weighting and fixed `C=0.01` logistic regression as
the primary attention-feature classifier. No hyperparameters are tuned.

The models are evaluated under the same five designs used for the
attention classifier: instance-level and prompt-grouped CV at `n=75`
and `n=127`, plus training on the initial 75 records and evaluation on
the later 52 records.

In [23]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    StratifiedGroupKFold,
    StratifiedKFold,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)


# ============================================================
# 1. Align Phase-2 records to the Phase-3 metadata order
# ============================================================

LABEL_TO_INT = {
    "hallucination_pivot": 0,
    "correction": 1,
}


def record_key(record):
    return (
        str(record["prompt_id"]),
        int(record["sample_idx"]),
    )


phase2_index = {}

for record in all_records:
    key = record_key(record)

    if key in phase2_index:
        raise ValueError(
            f"Duplicate Phase-2 record key: {key}"
        )

    phase2_index[key] = record


def align_phase2_records(metadata):
    aligned = []

    for meta_record in metadata:
        key = record_key(meta_record)

        if key not in phase2_index:
            raise KeyError(
                f"No Phase-2 record found for {key}"
            )

        record = phase2_index[key]

        if record["label"] != meta_record["label"]:
            raise ValueError(
                f"Label mismatch for {key}: "
                f"Phase 2={record['label']}, "
                f"metadata={meta_record['label']}"
            )

        aligned.append(record)

    return aligned


records_75 = align_phase2_records(meta_75)
records_127 = align_phase2_records(meta_127)

records_new = [
    record
    for record, is_new in zip(
        records_127,
        new_mask,
    )
    if is_new
]

assert len(records_75) == 75
assert len(records_127) == 127
assert len(records_new) == 52

assert np.array_equal(
    np.array(
        [
            LABEL_TO_INT[r["label"]]
            for r in records_75
        ]
    ),
    y75,
)

assert np.array_equal(
    np.array(
        [
            LABEL_TO_INT[r["label"]]
            for r in records_127
        ]
    ),
    y127,
)

assert np.array_equal(
    np.array(
        [
            LABEL_TO_INT[r["label"]]
            for r in records_new
        ]
    ),
    ynew,
)


# ============================================================
# 2. Recover only the generated text before the pivot
# ============================================================

def extract_pre_pivot_text(record):
    generated_text = str(
        record["generated_text"]
    )

    pivot_phrase = str(
        record["pivot_phrase"]
    )

    position = generated_text.casefold().find(
        pivot_phrase.casefold()
    )

    if position < 0:
        raise ValueError(
            "Pivot phrase not found in generated text: "
            f"{record_key(record)}, "
            f"phrase={pivot_phrase!r}"
        )

    return generated_text[:position].strip()


def normalize_phrase(value):
    value = str(value).casefold().strip()
    return re.sub(r"\s+", " ", value)


def safe_float(value, default=0.0):
    try:
        return float(value)
    except (TypeError, ValueError):
        return float(default)


def build_baseline_frame(records):
    rows = []

    for record in records:
        pre_text = extract_pre_pivot_text(
            record
        )

        pivot_start = safe_float(
            record.get("pivot_token_start")
        )

        pre_pivot_tokens = safe_float(
            record.get(
                "pre_pivot_tokens",
                pivot_start,
            )
        )

        total_tokens = safe_float(
            record.get(
                "total_tokens",
                max(pivot_start, 1),
            ),
            default=max(pivot_start, 1),
        )

        total_tokens = max(
            total_tokens,
            1.0,
        )

        rows.append(
            {
                "pre_pivot_text": pre_text,
                "category": str(
                    record.get(
                        "category",
                        "unknown",
                    )
                ),
                "pivot_type": str(
                    record.get(
                        "pivot_type",
                        "unknown",
                    )
                ),
                "pivot_phrase": normalize_phrase(
                    record.get(
                        "pivot_phrase",
                        "unknown",
                    )
                ),
                "prompt_id": str(
                    record["prompt_id"]
                ),
                "pre_pivot_tokens": (
                    pre_pivot_tokens
                ),
                "total_tokens": total_tokens,
                "pivot_fraction": (
                    pivot_start / total_tokens
                ),
                "pre_pivot_characters": float(
                    len(pre_text)
                ),
                "pre_pivot_words": float(
                    len(pre_text.split())
                ),
            }
        )

    return pd.DataFrame(rows)


frame_75 = build_baseline_frame(
    records_75
)

frame_127 = build_baseline_frame(
    records_127
)

frame_new = build_baseline_frame(
    records_new
)

assert len(frame_75) == len(y75)
assert len(frame_127) == len(y127)
assert len(frame_new) == len(ynew)

assert frame_75[
    "pre_pivot_text"
].str.len().gt(0).all()

assert frame_127[
    "pre_pivot_text"
].str.len().gt(0).all()

assert set(
    frame_75["prompt_id"]
).isdisjoint(
    set(frame_new["prompt_id"])
)


# ============================================================
# 3. Define baseline feature pipelines
# ============================================================

BASIC_CATEGORICAL_COLUMNS = [
    "category",
    "pivot_type",
    "pivot_phrase",
]

PROMPT_ID_CATEGORICAL_COLUMNS = (
    BASIC_CATEGORICAL_COLUMNS
    + ["prompt_id"]
)

NUMERIC_COLUMNS = [
    "pre_pivot_tokens",
    "total_tokens",
    "pivot_fraction",
    "pre_pivot_characters",
    "pre_pivot_words",
]


def make_one_hot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
        )
    except TypeError:
        # Compatibility with older scikit-learn.
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=True,
        )


def make_categorical_pipeline():
    return Pipeline(
        [
            (
                "impute",
                SimpleImputer(
                    strategy="most_frequent"
                ),
            ),
            (
                "one_hot",
                make_one_hot_encoder(),
            ),
        ]
    )


def make_numeric_pipeline():
    return Pipeline(
        [
            (
                "impute",
                SimpleImputer(
                    strategy="median"
                ),
            ),
            (
                "scale",
                StandardScaler(),
            ),
        ]
    )


def make_classifier():
    return LogisticRegression(
        class_weight="balanced",
        C=0.01,
        max_iter=3000,
        random_state=42,
    )


def make_metadata_pipeline(
    include_prompt_id=False,
):
    categorical_columns = (
        PROMPT_ID_CATEGORICAL_COLUMNS
        if include_prompt_id
        else BASIC_CATEGORICAL_COLUMNS
    )

    features = ColumnTransformer(
        [
            (
                "categorical",
                make_categorical_pipeline(),
                categorical_columns,
            ),
            (
                "numeric",
                make_numeric_pipeline(),
                NUMERIC_COLUMNS,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
    )

    return Pipeline(
        [
            ("features", features),
            ("classifier", make_classifier()),
        ]
    )


def make_text_pipeline():
    features = ColumnTransformer(
        [
            (
                "text",
                TfidfVectorizer(
                    lowercase=True,
                    strip_accents="unicode",
                    ngram_range=(1, 2),
                    min_df=2,
                    max_features=5000,
                    sublinear_tf=True,
                ),
                "pre_pivot_text",
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
    )

    return Pipeline(
        [
            ("features", features),
            ("classifier", make_classifier()),
        ]
    )


def make_text_metadata_pipeline():
    features = ColumnTransformer(
        [
            (
                "text",
                TfidfVectorizer(
                    lowercase=True,
                    strip_accents="unicode",
                    ngram_range=(1, 2),
                    min_df=2,
                    max_features=5000,
                    sublinear_tf=True,
                ),
                "pre_pivot_text",
            ),
            (
                "categorical",
                make_categorical_pipeline(),
                BASIC_CATEGORICAL_COLUMNS,
            ),
            (
                "numeric",
                make_numeric_pipeline(),
                NUMERIC_COLUMNS,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
    )

    return Pipeline(
        [
            ("features", features),
            ("classifier", make_classifier()),
        ]
    )


BASELINE_PIPELINES = {
    "metadata_basic": (
        make_metadata_pipeline(
            include_prompt_id=False
        )
    ),
    "metadata_plus_prompt_id": (
        make_metadata_pipeline(
            include_prompt_id=True
        )
    ),
    "text_tfidf": make_text_pipeline(),
    "text_plus_metadata": (
        make_text_metadata_pipeline()
    ),
}


# ============================================================
# 4. Evaluation helpers
# ============================================================

def calculate_metrics(
    y_true,
    probabilities,
    predictions,
):
    prevalence = float(
        np.mean(y_true)
    )

    average_precision = float(
        average_precision_score(
            y_true,
            probabilities,
        )
    )

    return {
        "n_records": int(len(y_true)),
        "n_corrections": int(
            np.sum(y_true)
        ),
        "correction_prevalence": (
            prevalence
        ),
        "roc_auc": float(
            roc_auc_score(
                y_true,
                probabilities,
            )
        ),
        "average_precision": (
            average_precision
        ),
        "ap_minus_prevalence": float(
            average_precision
            - prevalence
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                predictions,
            )
        ),
        "f1": float(
            f1_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
    }


def evaluate_oof(
    frame,
    y,
    pipeline,
    groups=None,
    grouped=False,
    seed=42,
):
    if grouped:
        if groups is None:
            raise ValueError(
                "groups are required "
                "for grouped CV"
            )

        splitter = StratifiedGroupKFold(
            n_splits=5,
            shuffle=True,
            random_state=seed,
        )

        split_iterator = splitter.split(
            frame,
            y,
            groups=groups,
        )
    else:
        splitter = StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=seed,
        )

        split_iterator = splitter.split(
            frame,
            y,
        )

    probabilities = np.zeros(
        len(y),
        dtype=float,
    )

    predictions = np.zeros(
        len(y),
        dtype=int,
    )

    assigned = np.zeros(
        len(y),
        dtype=bool,
    )

    for fold, (
        train_index,
        test_index,
    ) in enumerate(split_iterator):
        if grouped:
            train_prompts = set(
                groups[train_index]
            )

            test_prompts = set(
                groups[test_index]
            )

            overlap = (
                train_prompts
                & test_prompts
            )

            assert not overlap, (
                f"Prompt leakage in fold "
                f"{fold}: {overlap}"
            )

        fold_pipeline = clone(
            pipeline
        )

        fold_pipeline.fit(
            frame.iloc[train_index],
            y[train_index],
        )

        probabilities[test_index] = (
            fold_pipeline.predict_proba(
                frame.iloc[test_index]
            )[:, 1]
        )

        predictions[test_index] = (
            fold_pipeline.predict(
                frame.iloc[test_index]
            )
        )

        assigned[test_index] = True

    assert assigned.all()

    return calculate_metrics(
        y,
        probabilities,
        predictions,
    )


def evaluate_disjoint(
    train_frame,
    train_y,
    test_frame,
    test_y,
    pipeline,
):
    fitted_pipeline = clone(
        pipeline
    )

    fitted_pipeline.fit(
        train_frame,
        train_y,
    )

    probabilities = (
        fitted_pipeline.predict_proba(
            test_frame
        )[:, 1]
    )

    predictions = (
        fitted_pipeline.predict(
            test_frame
        )
    )

    return calculate_metrics(
        test_y,
        probabilities,
        predictions,
    )


# ============================================================
# 5. Run all four baseline families
# ============================================================

baseline_rows = []

for model_name, pipeline in (
    BASELINE_PIPELINES.items()
):
    evaluations = [
        (
            "Instance-level CV, n=75",
            evaluate_oof(
                frame_75,
                y75,
                pipeline,
                grouped=False,
                seed=42,
            ),
        ),
        (
            "Instance-level CV, n=127",
            evaluate_oof(
                frame_127,
                y127,
                pipeline,
                grouped=False,
                seed=42,
            ),
        ),
        (
            "Prompt-grouped CV, n=75",
            evaluate_oof(
                frame_75,
                y75,
                pipeline,
                groups=pid75,
                grouped=True,
                seed=42,
            ),
        ),
        (
            "Prompt-grouped CV, n=127",
            evaluate_oof(
                frame_127,
                y127,
                pipeline,
                groups=pid127,
                grouped=True,
                seed=42,
            ),
        ),
        (
            "Train n=75, test later n=52",
            evaluate_disjoint(
                frame_75,
                y75,
                frame_new,
                ynew,
                pipeline,
            ),
        ),
    ]

    for evaluation_name, metrics in (
        evaluations
    ):
        baseline_rows.append(
            {
                "model": model_name,
                "evaluation": (
                    evaluation_name
                ),
                **metrics,
            }
        )


baseline_df = pd.DataFrame(
    baseline_rows
)

assert len(baseline_df) == 20


# ============================================================
# 6. Print compact comparison
# ============================================================

print(
    "\nText and metadata baseline results:\n"
)

print(
    baseline_df[
        [
            "model",
            "evaluation",
            "roc_auc",
            "average_precision",
            "correction_prevalence",
            "balanced_accuracy",
            "f1",
        ]
    ].to_string(
        index=False,
        formatters={
            "roc_auc": (
                lambda x: f"{x:.3f}"
            ),
            "average_precision": (
                lambda x: f"{x:.3f}"
            ),
            "correction_prevalence": (
                lambda x: f"{x:.3f}"
            ),
            "balanced_accuracy": (
                lambda x: f"{x:.3f}"
            ),
            "f1": (
                lambda x: f"{x:.3f}"
            ),
        },
    )
)


# ============================================================
# 7. Save machine-readable results
# ============================================================

BASELINE_CSV_PATH = (
    Path(OUTPUT_DIR)
    / "qwen_text_metadata_baselines.csv"
)

BASELINE_JSON_PATH = (
    Path(OUTPUT_DIR)
    / "qwen_text_metadata_baselines.json"
)

baseline_df.to_csv(
    BASELINE_CSV_PATH,
    index=False,
)

baseline_export = {
    "analysis": (
        "Qwen pre-pivot text and "
        "metadata baselines"
    ),
    "positive_class": "correction",
    "text_scope": (
        "Generated text occurring before "
        "the detected pivot phrase"
    ),
    "models": {
        "metadata_basic": {
            "categorical": (
                BASIC_CATEGORICAL_COLUMNS
            ),
            "numeric": NUMERIC_COLUMNS,
            "includes_prompt_id": False,
        },
        "metadata_plus_prompt_id": {
            "categorical": (
                PROMPT_ID_CATEGORICAL_COLUMNS
            ),
            "numeric": NUMERIC_COLUMNS,
            "includes_prompt_id": True,
            "purpose": (
                "Explicit prompt-identity "
                "leakage diagnostic"
            ),
        },
        "text_tfidf": {
            "ngram_range": [1, 2],
            "min_df": 2,
            "max_features": 5000,
        },
        "text_plus_metadata": {
            "text": (
                "Same TF-IDF configuration"
            ),
            "categorical": (
                BASIC_CATEGORICAL_COLUMNS
            ),
            "numeric": NUMERIC_COLUMNS,
            "includes_prompt_id": False,
        },
    },
    "classifier": {
        "model": "LogisticRegression",
        "class_weight": "balanced",
        "C": 0.01,
        "max_iter": 3000,
        "random_state": 42,
        "hyperparameter_tuning": False,
    },
    "cv_split_seed": 42,
    "results": (
        baseline_df.to_dict(
            orient="records"
        )
    ),
}

with BASELINE_JSON_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        baseline_export,
        file,
        indent=2,
    )

print(
    "\nSaved:",
    BASELINE_CSV_PATH,
)

print(
    "Saved:",
    BASELINE_JSON_PATH,
)


Text and metadata baseline results:

                  model                  evaluation roc_auc average_precision correction_prevalence balanced_accuracy    f1
         metadata_basic     Instance-level CV, n=75   0.573             0.226                 0.187             0.559 0.308
         metadata_basic    Instance-level CV, n=127   0.465             0.168                 0.173             0.517 0.268
         metadata_basic     Prompt-grouped CV, n=75   0.585             0.247                 0.187             0.567 0.316
         metadata_basic    Prompt-grouped CV, n=127   0.537             0.224                 0.173             0.535 0.286
         metadata_basic Train n=75, test later n=52   0.491             0.236                 0.154             0.517 0.269
metadata_plus_prompt_id     Instance-level CV, n=75   0.587             0.235                 0.187             0.567 0.316
metadata_plus_prompt_id    Instance-level CV, n=127   0.475             0.172                 

In [24]:
combined_check = clone(
    BASELINE_PIPELINES["text_plus_metadata"]
)

combined_check.fit(
    frame_75,
    y75,
)

feature_transformer = (
    combined_check.named_steps["features"]
)

transformed = feature_transformer.transform(
    frame_75
)

classifier = combined_check.named_steps[
    "classifier"
]

print("Combined feature matrix shape:", transformed.shape)
print(
    "Total nonzero coefficients:",
    int(np.sum(np.abs(classifier.coef_[0]) > 1e-12)),
)

feature_names = (
    feature_transformer.get_feature_names_out()
)

text_mask = np.array(
    [
        name.startswith("text__")
        for name in feature_names
    ]
)

metadata_mask = ~text_mask

print("TF-IDF feature count:", int(text_mask.sum()))
print(
    "Nonzero TF-IDF coefficients:",
    int(
        np.sum(
            np.abs(
                classifier.coef_[0][text_mask]
            ) > 1e-12
        )
    ),
)

print(
    "TF-IDF coefficient L1 norm:",
    float(
        np.abs(
            classifier.coef_[0][text_mask]
        ).sum()
    ),
)

print(
    "Metadata coefficient L1 norm:",
    float(
        np.abs(
            classifier.coef_[0][metadata_mask]
        ).sum()
    ),
)

Combined feature matrix shape: (75, 1466)
Total nonzero coefficients: 1466
TF-IDF feature count: 1447
Nonzero TF-IDF coefficients: 1447
TF-IDF coefficient L1 norm: 1.7917480875075495
Metadata coefficient L1 norm: 0.3101387091392235


## Non-corrective-pivot subtype audit

The binary `hallucination_pivot` class combines at least two
behaviorally different outcomes:

- right-to-wrong: a correct pre-pivot answer is abandoned;
- wrong-to-wrong: reasoning remains incorrect after the pivot.

Subtype assignments are reconstructed conservatively from the archived
`label_reason` field. Explicit standardized reasons and clear manual
descriptions are assigned automatically. Stylistic/right-to-right and
unresolved records are reported separately rather than forced into one
of the two principal subtypes.

In [25]:
import json
import re
from collections import Counter
from pathlib import Path

import pandas as pd


def normalize_reason(value):
    text = str(value or "").casefold()
    text = text.replace("→", "-to-")
    text = text.replace("_", " ")
    return re.sub(r"\s+", " ", text).strip()


def classify_hallucination_subtype(record):
    """
    Conservatively reconstruct the subtype of a record labeled
    hallucination_pivot from its archived label_reason.

    Returns:
        subtype, rule, confidence
    """
    if record.get("label") != "hallucination_pivot":
        return "not_applicable", "non_hallucination_label", "exact"

    reason = normalize_reason(
        record.get("label_reason")
    )

    # --------------------------------------------------------
    # Exact standardized automatic-label reasons
    # --------------------------------------------------------
    if reason == (
        "correct answer present before pivot "
        "but lost after"
    ):
        return (
            "right_to_wrong",
            "standard_automatic_reason",
            "exact",
        )

    if reason == "wrong both before and after pivot":
        return (
            "wrong_to_wrong",
            "standard_automatic_reason",
            "exact",
        )

    # Gemma-style standardized spelling, retained for safety.
    if reason == "wrong then wrong":
        return (
            "wrong_to_wrong",
            "standard_automatic_reason",
            "exact",
        )

    # --------------------------------------------------------
    # Explicit language in manually written reasons
    # --------------------------------------------------------
    if (
        "right-to-wrong" in reason
        or "right to wrong" in reason
    ):
        return (
            "right_to_wrong",
            "explicit_manual_phrase",
            "exact",
        )

    if (
        "wrong-to-wrong" in reason
        or "wrong to wrong" in reason
    ):
        return (
            "wrong_to_wrong",
            "explicit_manual_phrase",
            "exact",
        )

    if (
        "right-to-right" in reason
        or "right to right" in reason
        or "correct answer present both sides" in reason
        or "stylistic pivot" in reason
    ):
        return (
            "right_to_right_or_stylistic",
            "explicit_manual_phrase",
            "exact",
        )

    # --------------------------------------------------------
    # Conservative narrative-pattern rules
    # --------------------------------------------------------
    right_to_wrong_patterns = [
        (
            r"correct(?:ly)? .{0,100} before "
            r".{0,180}(?:incorrect|wrong|reject|lost)"
        ),
        (
            r"before the pivot .{0,100}correct"
            r".{0,180}after the pivot"
            r".{0,100}(?:incorrect|wrong|reject)"
        ),
        (
            r"correct result .{0,120}"
            r"(?:reject|abandon|lost|incorrect)"
        ),
        (
            r"valid .{0,100}(?:solution|answer|result)"
            r".{0,120}(?:reject|incorrect|wrong)"
        ),
    ]

    for pattern in right_to_wrong_patterns:
        if re.search(pattern, reason):
            return (
                "right_to_wrong",
                f"narrative_regex:{pattern}",
                "heuristic",
            )

    wrong_to_wrong_patterns = [
        (
            r"(?:wrong|incorrect) .{0,100} before"
            r".{0,180}(?:wrong|incorrect) .{0,100} after"
        ),
        (
            r"begins? with .{0,80}(?:wrong|incorrect)"
            r".{0,180}continues? with .{0,80}"
            r"(?:wrong|incorrect)"
        ),
        (
            r"continues? with incorrect"
        ),
        (
            r"never reaches? .{0,60}"
            r"(?:the )?correct"
        ),
        (
            r"remains? incorrect"
        ),
    ]

    for pattern in wrong_to_wrong_patterns:
        if re.search(pattern, reason):
            return (
                "wrong_to_wrong",
                f"narrative_regex:{pattern}",
                "heuristic",
            )

    return (
        "unresolved_manual_review",
        "no_conservative_rule_matched",
        "unresolved",
    )


def build_subtype_rows(
    records,
    sample_name,
):
    rows = []

    for record in records:
        if (
            record.get("label")
            != "hallucination_pivot"
        ):
            continue

        subtype, rule, confidence = (
            classify_hallucination_subtype(
                record
            )
        )

        rows.append(
            {
                "sample": sample_name,
                "prompt_id": str(
                    record["prompt_id"]
                ),
                "sample_idx": int(
                    record["sample_idx"]
                ),
                "category": str(
                    record.get(
                        "category",
                        "unknown",
                    )
                ),
                "label": str(
                    record["label"]
                ),
                "label_method": str(
                    record.get(
                        "label_method",
                        "unknown",
                    )
                ),
                "human_verified": bool(
                    record.get(
                        "human_verified",
                        False,
                    )
                ),
                "subtype": subtype,
                "classification_rule": rule,
                "classification_confidence": (
                    confidence
                ),
                "label_reason": str(
                    record.get(
                        "label_reason",
                        ""
                    )
                ),
            }
        )

    return rows


subtype_rows = []

for sample_name, sample_records in [
    ("initial_n75", records_75),
    ("expanded_n127", records_127),
    ("later_n52", records_new),
]:
    subtype_rows.extend(
        build_subtype_rows(
            sample_records,
            sample_name,
        )
    )


subtype_df = pd.DataFrame(
    subtype_rows
)

expected_hallucination_counts = {
    "initial_n75": 61,
    "expanded_n127": 105,
    "later_n52": 44,
}

for sample_name, expected_count in (
    expected_hallucination_counts.items()
):
    actual_count = int(
        (
            subtype_df["sample"]
            == sample_name
        ).sum()
    )

    assert actual_count == expected_count, (
        f"{sample_name}: expected "
        f"{expected_count} hallucination "
        f"records, found {actual_count}"
    )


summary_rows = []

for sample_name in [
    "initial_n75",
    "expanded_n127",
    "later_n52",
]:
    sample_df = subtype_df[
        subtype_df["sample"]
        == sample_name
    ]

    counts = Counter(
        sample_df["subtype"]
    )

    total = len(sample_df)

    for subtype in [
        "right_to_wrong",
        "wrong_to_wrong",
        "right_to_right_or_stylistic",
        "unresolved_manual_review",
    ]:
        count = int(
            counts.get(subtype, 0)
        )

        summary_rows.append(
            {
                "sample": sample_name,
                "subtype": subtype,
                "count": count,
                "proportion": (
                    count / total
                    if total
                    else float("nan")
                ),
                "total_hallucination_pivots": (
                    total
                ),
            }
        )


subtype_summary_df = pd.DataFrame(
    summary_rows
)

print(
    "\nNon-corrective-pivot subtype "
    "counts:\n"
)

print(
    subtype_summary_df.pivot(
        index="subtype",
        columns="sample",
        values="count",
    )
    .fillna(0)
    .astype(int)
    .to_string()
)


unresolved_df = subtype_df[
    subtype_df["subtype"].isin(
        [
            "right_to_right_or_stylistic",
            "unresolved_manual_review",
        ]
    )
].copy()

print(
    "\nRecords requiring confirmation "
    "or separate reporting:",
    len(unresolved_df),
)

if not unresolved_df.empty:
    print(
        unresolved_df[
            [
                "sample",
                "prompt_id",
                "sample_idx",
                "subtype",
                "label_reason",
            ]
        ]
        .drop_duplicates(
            subset=[
                "prompt_id",
                "sample_idx",
            ]
        )
        .to_string(
            index=False,
            max_colwidth=120,
        )
    )


SUBTYPE_DETAIL_PATH = (
    Path(OUTPUT_DIR)
    / "qwen_hallucination_subtype_audit.csv"
)

SUBTYPE_SUMMARY_PATH = (
    Path(OUTPUT_DIR)
    / "qwen_hallucination_subtype_summary.json"
)

SUBTYPE_REVIEW_PATH = (
    Path(OUTPUT_DIR)
    / "qwen_hallucination_subtype_manual_review.csv"
)

subtype_df.to_csv(
    SUBTYPE_DETAIL_PATH,
    index=False,
)

unresolved_df.drop_duplicates(
    subset=[
        "prompt_id",
        "sample_idx",
    ]
).to_csv(
    SUBTYPE_REVIEW_PATH,
    index=False,
)

subtype_export = {
    "analysis": (
        "Conservative reconstruction of "
        "hallucination-pivot subtypes"
    ),
    "principal_subtypes": {
        "right_to_wrong": (
            "Correct pre-pivot answer is "
            "abandoned after the pivot."
        ),
        "wrong_to_wrong": (
            "Reasoning is incorrect before "
            "and remains incorrect after "
            "the pivot."
        ),
    },
    "note": (
        "Stylistic/right-to-right and "
        "unresolved records are not forced "
        "into a principal subtype."
    ),
    "summary": (
        subtype_summary_df.to_dict(
            orient="records"
        )
    ),
}

with SUBTYPE_SUMMARY_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        subtype_export,
        file,
        indent=2,
    )

print("\nSaved:", SUBTYPE_DETAIL_PATH)
print("Saved:", SUBTYPE_SUMMARY_PATH)
print("Saved:", SUBTYPE_REVIEW_PATH)


Non-corrective-pivot subtype counts:

sample                       expanded_n127  initial_n75  later_n52
subtype                                                           
right_to_right_or_stylistic             28           13         15
right_to_wrong                          38           25         13
unresolved_manual_review                 0            0          0
wrong_to_wrong                          39           23         16

Records requiring confirmation or separate reporting: 56
       sample prompt_id  sample_idx                     subtype                                               label_reason
  initial_n75    mis_06          11 right_to_right_or_stylistic correct answer present both sides — likely stylistic pivot
  initial_n75    mis_06          14 right_to_right_or_stylistic correct answer present both sides — likely stylistic pivot
  initial_n75    mis_10          12 right_to_right_or_stylistic correct answer present both sides — likely stylistic pivot
  initial

In [26]:
# ============================================================
# Verify the labeling policy for right-to-right/stylistic pivots
# ============================================================

from pathlib import Path
import pandas as pd


def key_of(record):
    return (
        str(record["prompt_id"]),
        int(record["sample_idx"]),
    )


initial_keys = {
    key_of(record)
    for record in records_75
}

later_keys = {
    key_of(record)
    for record in records_new
}


right_right_unique = (
    subtype_df[
        subtype_df["subtype"]
        == "right_to_right_or_stylistic"
    ]
    .drop_duplicates(
        subset=[
            "prompt_id",
            "sample_idx",
        ]
    )
    .copy()
)


def assign_round(row):
    key = (
        str(row["prompt_id"]),
        int(row["sample_idx"]),
    )

    if key in initial_keys:
        return "initial"

    if key in later_keys:
        return "later"

    return "unknown"


right_right_unique["collection_round"] = (
    right_right_unique.apply(
        assign_round,
        axis=1,
    )
)


print("=" * 70)
print("UNIQUE RIGHT-TO-RIGHT / STYLISTIC RECORDS")
print("=" * 70)

print(
    "Unique records:",
    len(right_right_unique),
)

print(
    "\nBy collection round:"
)

print(
    right_right_unique[
        "collection_round"
    ].value_counts()
)


print(
    "\nBy label method:"
)

print(
    right_right_unique[
        "label_method"
    ].value_counts(
        dropna=False
    )
)


print(
    "\nBy human_verified:"
)

print(
    right_right_unique[
        "human_verified"
    ].value_counts(
        dropna=False
    )
)


print(
    "\nLabel method × human verification:"
)

print(
    pd.crosstab(
        right_right_unique[
            "label_method"
        ],
        right_right_unique[
            "human_verified"
        ],
        dropna=False,
    )
)


# ------------------------------------------------------------
# Compare against records currently labeled uncertain
# ------------------------------------------------------------

uncertain_records = [
    record
    for record in all_records
    if record.get("label") == "uncertain"
]

uncertain_reason_rows = []

for record in uncertain_records:
    reason = str(
        record.get(
            "label_reason",
            "",
        )
    ).casefold()

    uncertain_reason_rows.append(
        {
            "prompt_id": record["prompt_id"],
            "sample_idx": record["sample_idx"],
            "label_reason": record.get(
                "label_reason",
                "",
            ),
            "human_verified": record.get(
                "human_verified",
                False,
            ),
            "label_method": record.get(
                "label_method",
                "",
            ),
            "mentions_correct_both_sides": (
                "correct answer present both sides"
                in reason
                or "right-to-right" in reason
                or "right to right" in reason
            ),
        }
    )


uncertain_reason_df = pd.DataFrame(
    uncertain_reason_rows
)

print(
    "\nUncertain records:",
    len(uncertain_reason_df),
)

if not uncertain_reason_df.empty:
    print(
        "Uncertain records whose reason "
        "mentions right-to-right:",
        int(
            uncertain_reason_df[
                "mentions_correct_both_sides"
            ].sum()
        ),
    )

    print(
        "\nUncertain label methods:"
    )

    print(
        uncertain_reason_df[
            "label_method"
        ].value_counts(
            dropna=False
        )
    )


# ------------------------------------------------------------
# Build a compact review file with the text around each pivot
# ------------------------------------------------------------

review_rows = []

for _, subtype_row in (
    right_right_unique.iterrows()
):
    key = (
        str(subtype_row["prompt_id"]),
        int(subtype_row["sample_idx"]),
    )

    record = phase2_index[key]

    generated_text = str(
        record["generated_text"]
    )

    pivot_phrase = str(
        record["pivot_phrase"]
    )

    pivot_position = (
        generated_text.casefold().find(
            pivot_phrase.casefold()
        )
    )

    if pivot_position >= 0:
        context_start = max(
            0,
            pivot_position - 300,
        )

        context_end = min(
            len(generated_text),
            pivot_position + 500,
        )

        pivot_context = (
            generated_text[
                context_start:context_end
            ]
        )
    else:
        pivot_context = generated_text[:800]

    review_rows.append(
        {
            "collection_round": (
                subtype_row[
                    "collection_round"
                ]
            ),
            "prompt_id": record[
                "prompt_id"
            ],
            "sample_idx": record[
                "sample_idx"
            ],
            "category": record[
                "category"
            ],
            "label": record[
                "label"
            ],
            "label_method": record.get(
                "label_method",
                "",
            ),
            "human_verified": record.get(
                "human_verified",
                False,
            ),
            "label_reason": record.get(
                "label_reason",
                "",
            ),
            "prompt_text": record.get(
                "prompt_text",
                "",
            ),
            "pivot_phrase": pivot_phrase,
            "pivot_context": pivot_context,
        }
    )


right_right_review_df = pd.DataFrame(
    review_rows
)

RIGHT_RIGHT_REVIEW_PATH = (
    Path(OUTPUT_DIR)
    / "qwen_right_to_right_policy_review.csv"
)

right_right_review_df.to_csv(
    RIGHT_RIGHT_REVIEW_PATH,
    index=False,
)

print(
    "\nSaved:",
    RIGHT_RIGHT_REVIEW_PATH,
)

UNIQUE RIGHT-TO-RIGHT / STYLISTIC RECORDS
Unique records: 28

By collection round:
collection_round
later      15
initial    13
Name: count, dtype: int64

By label method:
label_method
human    28
Name: count, dtype: int64

By human_verified:
human_verified
True    28
Name: count, dtype: int64

Label method × human verification:
human_verified  True
label_method        
human             28

Uncertain records: 22
Uncertain records whose reason mentions right-to-right: 22

Uncertain label methods:
label_method
automatic             8
human                 7
corrected_keywords    7
Name: count, dtype: int64

Saved: /kaggle/working/pivot_research/qwen_right_to_right_policy_review.csv


## Blinded right-to-right label-consistency review

Both the `hallucination_pivot` and `uncertain` classes contain records
described as correct before and after the pivot. To verify that the
distinction reflects a reproducible annotation policy rather than an
inconsistent label boundary, all unique right-to-right records are
exported for blinded re-review.

The review sheet hides the existing label, label method, and label
reason. Each record should be assigned one of:

- `non_corrective_clear`: a clearly unnecessary, stylistic, or misleading
  pivot that does not improve the answer;
- `uncertain_ambiguous`: the correctness or functional role of the pivot
  cannot be determined confidently;
- `correction`: the pivot genuinely repairs an earlier error;
- `not_a_pivot`: the detected phrase does not function as a reasoning pivot.

In [29]:
import re
from pathlib import Path

import numpy as np
import pandas as pd


def normalized_reason(record):
    return re.sub(
        r"\s+",
        " ",
        str(record.get("label_reason", "")).casefold(),
    ).strip()


def is_right_to_right_candidate(record):
    label = str(record.get("label", ""))

    if label not in {
        "hallucination_pivot",
        "uncertain",
    }:
        return False

    reason = normalized_reason(record)

    return (
        "correct answer present both sides"
        in reason
    )

def split_around_pivot(record):
    generated_text = str(record["generated_text"])
    pivot_phrase = str(record["pivot_phrase"])

    pivot_position = generated_text.casefold().find(
        pivot_phrase.casefold()
    )

    if pivot_position < 0:
        return (
            generated_text,
            "",
            "",
            False,
        )

    pre_text = generated_text[:pivot_position].strip()

    post_start = pivot_position + len(pivot_phrase)
    post_text = generated_text[post_start:].strip()

    return (
        pre_text,
        pivot_phrase,
        post_text,
        True,
    )


# Collect unique records only.
candidate_records = {}

for record in all_records:
    if not is_right_to_right_candidate(record):
        continue

    key = (
        str(record["prompt_id"]),
        int(record["sample_idx"]),
    )

    candidate_records[key] = record


assert len(candidate_records) == 50, (
    "Expected 50 unique right-to-right candidates, "
    f"found {len(candidate_records)}."
)


review_rows = []
key_rows = []

for key, record in candidate_records.items():
    (
        pre_text,
        pivot_phrase,
        post_text,
        pivot_found,
    ) = split_around_pivot(record)

    review_rows.append(
        {
            "prompt_id": str(record["prompt_id"]),
            "sample_idx": int(record["sample_idx"]),
            "category": str(record["category"]),
            "prompt_text": str(record["prompt_text"]),
            "pre_pivot_text": pre_text,
            "pivot_phrase": pivot_phrase,
            "post_pivot_text": post_text,
            "pivot_found": bool(pivot_found),

            # Fill these manually.
            "reviewer_decision": "",
            "reviewer_confidence": "",
            "reviewer_notes": "",
        }
    )

    key_rows.append(
        {
            "prompt_id": str(record["prompt_id"]),
            "sample_idx": int(record["sample_idx"]),
            "existing_label": str(record["label"]),
            "existing_label_method": str(
                record.get("label_method", "")
            ),
            "existing_human_verified": bool(
                record.get("human_verified", False)
            ),
            "existing_label_reason": str(
                record.get("label_reason", "")
            ),
        }
    )


review_df = pd.DataFrame(review_rows)
key_df = pd.DataFrame(key_rows)


# Randomize review order while keeping a separate answer key.
rng = np.random.default_rng(42)
review_order = rng.permutation(len(review_df))

review_df = review_df.iloc[review_order].reset_index(drop=True)
review_df.insert(
    0,
    "review_id",
    [
        f"RTR_{index:03d}"
        for index in range(1, len(review_df) + 1)
    ],
)


# Merge review IDs into the hidden key.
key_df = key_df.merge(
    review_df[
        [
            "review_id",
            "prompt_id",
            "sample_idx",
        ]
    ],
    on=[
        "prompt_id",
        "sample_idx",
    ],
    how="left",
    validate="one_to_one",
)

key_df = key_df[
    [
        "review_id",
        "prompt_id",
        "sample_idx",
        "existing_label",
        "existing_label_method",
        "existing_human_verified",
        "existing_label_reason",
    ]
].sort_values("review_id")


assert review_df["pivot_found"].all()
assert review_df["review_id"].is_unique
assert key_df["review_id"].notna().all()


BLINDED_REVIEW_PATH = (
    Path(OUTPUT_DIR)
    / "qwen_right_to_right_blinded_review.csv"
)

HIDDEN_KEY_PATH = (
    Path(OUTPUT_DIR)
    / "qwen_right_to_right_existing_label_key.csv"
)

review_df.to_csv(
    BLINDED_REVIEW_PATH,
    index=False,
)

key_df.to_csv(
    HIDDEN_KEY_PATH,
    index=False,
)


print("=" * 70)
print("BLINDED RIGHT-TO-RIGHT REVIEW")
print("=" * 70)

print("Unique records:", len(review_df))
print("Pivot phrase found:", int(review_df["pivot_found"].sum()))

print("\nExisting labels, hidden in review sheet:")
print(key_df["existing_label"].value_counts())

print("\nSaved blinded review:")
print(BLINDED_REVIEW_PATH)

print("\nSaved hidden existing-label key:")
print(HIDDEN_KEY_PATH)

BLINDED RIGHT-TO-RIGHT REVIEW
Unique records: 50
Pivot phrase found: 50

Existing labels, hidden in review sheet:
existing_label
hallucination_pivot    28
uncertain              22
Name: count, dtype: int64

Saved blinded review:
/kaggle/working/pivot_research/qwen_right_to_right_blinded_review.csv

Saved hidden existing-label key:
/kaggle/working/pivot_research/qwen_right_to_right_existing_label_key.csv
